# 05장 보안 실습 — SSH 로그 파이프라인


## Goal

원문→실패 행→주소→빈도를 단계별로 확인합니다.

[교안과 분석 질문](../../05-text-processing/05-4-auth-pipeline.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-05-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'auth.log': '2026-09-10T09:01:00+09:00 lab-web-01 sshd[101]: Failed password for invalid user guest from 192.0.2.10 port 50100 ssh2\n2026-09-10T09:01:10+09:00 lab-web-01 sshd[102]: Failed password for analyst from 192.0.2.10 port 50101 ssh2\n2026-09-10T09:01:20+09:00 lab-web-01 sshd[103]: Failed password for analyst from 198.51.100.8 port 50102 ssh2\n2026-09-10T09:02:00+09:00 lab-web-01 sshd[104]: Accepted publickey for analyst from 192.0.2.10 port 50103 ssh2\n2026-09-10T09:03:00+09:00 lab-web-01 sudo: analyst : TTY=pts/0 ; PWD=/home/analyst ; USER=root ; COMMAND=/usr/bin/id\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 실패 3건, 주소별 2/1건, 공개키 성공 1건, sudo 기록 1건

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 원본에서 실패 행 선택


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'Failed password' "$COURSE_DATA/auth.log" > "$COURSE_OUT/failed.txt"
test "$(wc -l < "$COURSE_OUT/failed.txt")" -eq 3
printf 'failed_rows=3\n'


### 2. from 표식 뒤의 주소 추출


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk '{for (i=1; i<NF; i++) if ($i=="from") {print $(i+1); break}}' \
    "$COURSE_OUT/failed.txt" > "$COURSE_OUT/addresses.txt"
test "$(wc -l < "$COURSE_OUT/addresses.txt")" -eq 3
cat "$COURSE_OUT/addresses.txt"


### 3. 정렬과 중복 집계


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
LC_ALL=C sort "$COURSE_OUT/addresses.txt" | uniq -c | LC_ALL=C sort -nr > "$COURSE_OUT/counts.txt"
grep -Eq '^[[:space:]]*2 192\.0\.2\.10$' "$COURSE_OUT/counts.txt"
grep -Eq '^[[:space:]]*1 198\.51\.100\.8$' "$COURSE_OUT/counts.txt"
cat "$COURSE_OUT/counts.txt"


### 4. 성공과 실패를 별도로 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'Accepted publickey' "$COURSE_DATA/auth.log" > "$COURSE_OUT/accepted.txt"
test "$(wc -l < "$COURSE_OUT/accepted.txt")" -eq 1
grep -F 'sudo:' "$COURSE_DATA/auth.log" > "$COURSE_OUT/sudo.txt"
test "$(wc -l < "$COURSE_OUT/sudo.txt")" -eq 1
printf 'accepted_publickey=1 sudo_records=1\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
